# Corrected official CODI answer-cue endpoint TSV-C experiment

This notebook runs the source-faithful correction. Both teacher and student activations are gathered at the colon in `The answer is:`. The primary scope includes the embedding output plus all 12 GPT-2 block outputs. A mandatory native loss-and-gradient parity gate runs before calibration.

Use Kaggle **Save Version → Save & Run All** with Internet and a T4 or newer GPU. The historical pre-cue dataset is not a valid resume input for this corrected contract.

## 1. Configuration

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit printed by setup.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""  # Optional passed full-GSM8K summary.json.
RESUME_INPUT = ""  # Optional prior *corrected* experiment export root.
RUN_REPRODUCTION_GATE_IF_MISSING = True
RUN_SMOKE = True
RUN_FULL_CALIBRATION = True
RUN_ALL_STATES_UTILITY = True
RUN_LAYER11_UTILITY = True

CALIBRATION_EXAMPLES = 5000
UPDATE_EXAMPLES = 256
VALIDATION_EXAMPLES = 256
CALIBRATION_BATCH_SIZE = 16
UTILITY_BATCH_SIZE = 4
SAMPLING_SEED = 11
RANDOM_BASIS_SEED = 20260803
RANK = 77
RELATIVE_UPDATE_NORM = 1e-4
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-endpoint-tsvc-corrected"

## 2. Install and pin the repository

In [ ]:
import datetime, hashlib, json, os, pathlib, shutil, subprocess, sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL SAVE VERSION RUN:", commit)

## 3. Hardware and corrected implementation checks

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "capability:", capability)
assert capability >= (7, 0), "Use a T4 or newer GPU with this PyTorch build"
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_official_codi.py",
    "tests/test_official_codi_target_utility.py",
    "tests/test_endpoint_tsvc.py",
    "tests/test_endpoint_tsvc_corrected.py",
    "tests/test_official_codi_endpoint_tsvc_analysis.py",
    "tests/test_official_codi_endpoint_tsvc_corrected_analysis.py",
], cwd=REPO_DIR, check=True)

## 4. Durable paths, logs, and optional corrected resume

In [ ]:
OUTPUT_ROOT = repo / "outputs" / "official_codi_endpoint_tsvc_corrected"
REPORT_ROOT = repo / "reports" / "official_codi_endpoint_tsvc_corrected"
LOG_ROOT = repo / "logs" / "official_codi_endpoint_tsvc_corrected"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT, VALIDATION_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}; inspect {log_path}")
    return log_path

if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(resume_root.rglob("official_codi_endpoint_tsvc_corrected/calibration_seed11/run_manifest.json"))
    assert manifests, "No corrected endpoint TSV-C calibration manifest found. Do not attach the historical pre-cue dataset."
    request_hashes = {json.loads(path.read_text()).get("request_sha256") for path in manifests}
    assert len(request_hashes) == 1, f"Incompatible corrected resume trees: {manifests}"
    source = sorted(manifests, key=lambda p: (len(p.parts), p.as_posix()))[0].parents[1]
    shutil.copytree(source, OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored corrected outputs from:", source)
else:
    print("Starting without prior corrected outputs")

## 5. Locate or create the official 43.67 percent reproduction gate

In [ ]:
EXPECTED_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"
def passed_summary(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate.get("status") if isinstance(gate, dict) else gate
    count = payload.get("evaluated_counts", {}).get("gsm8k")
    revision = payload.get("checkpoint_revision")
    return status == "passed" and count == 1319 and revision in {None, EXPECTED_REVISION}

if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT)
    assert passed_summary(REPRODUCTION_SUMMARY)
else:
    candidates = [p for p in pathlib.Path("/kaggle/input").rglob("summary.json") if passed_summary(p)]
    candidates += [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    if not candidates:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", "configs/official_codi_gpt2.yaml", "--datasets", "gsm8k", "--limit", "0", "--device", "cuda", "--output-dir", str(VALIDATION_ROOT)], "official_codi_gsm8k_gate.log")
        candidates = [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    assert candidates, "A passed full-GSM8K official CODI summary is required"
    REPRODUCTION_SUMMARY = sorted(candidates, key=lambda p: p.as_posix())[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 6. Commands and mandatory corrected smoke test

In [ ]:
def collection_command(root, calibration, update, validation, batch):
    return [sys.executable, "-u", "scripts/collect_official_codi_endpoint_tsvc_corrected.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--output-dir", str(root), "--calibration-examples", str(calibration), "--update-examples", str(update), "--validation-examples", str(validation), "--batch-size", str(batch), "--save-every", "8", "--rank", str(RANK), "--sampling-seed", str(SAMPLING_SEED), "--random-basis-seed", str(RANDOM_BASIS_SEED), "--parity-examples", "4", "--precision", PRECISION, "--device", "cuda"]

def utility_command(root, basis, scope, bootstrap_samples=BOOTSTRAP_SAMPLES):
    return [sys.executable, "-u", "scripts/run_official_codi_endpoint_tsvc_corrected_utility.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--basis", str(basis), "--output-dir", str(root), "--scope", scope, "--batch-size", str(UTILITY_BATCH_SIZE), "--relative-update-norm", str(RELATIVE_UPDATE_NORM), "--precision", PRECISION, "--device", "cuda", "--seed", str(SAMPLING_SEED), "--bootstrap-samples", str(bootstrap_samples), "--bootstrap-seed", str(BOOTSTRAP_SEED)]

SMOKE_CALIBRATION = OUTPUT_ROOT / "smoke_seed11" / "calibration"
SMOKE_UTILITY = OUTPUT_ROOT / "smoke_seed11" / "endpoint_all_states"
if RUN_SMOKE:
    run_persisted(collection_command(SMOKE_CALIBRATION, 16, 8, 8, 8), "smoke_corrected_calibration.log")
    parity = json.loads((SMOKE_CALIBRATION / "native_loss_gradient_parity.json").read_text())
    assert parity["status"] == "passed", parity
    assert parity["teacher_shape"][1:] == [13, 768]
    assert parity["student_shape"] == parity["teacher_shape"]
    run_persisted(utility_command(SMOKE_UTILITY, SMOKE_CALIBRATION / "basis.pt", "endpoint_all_states", 500), "smoke_corrected_utility.log")
    assert json.loads((SMOKE_UTILITY / "run_manifest.json").read_text())["state"] == "complete"
    print("Corrected parity and utility smoke path complete. Smoke gate is diagnostic only.")
else:
    print("Smoke skipped. Do not run full calibration unless an equivalent corrected parity artifact already passed.")

## 7. Fit or resume the 5,000-example corrected bases

In [ ]:
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration_seed11"
BASIS_PATH = CALIBRATION_ROOT / "basis.pt"
if RUN_FULL_CALIBRATION:
    command = collection_command(CALIBRATION_ROOT, CALIBRATION_EXAMPLES, UPDATE_EXAMPLES, VALIDATION_EXAMPLES, CALIBRATION_BATCH_SIZE)
    command[command.index("--save-every") + 1] = "500"
    run_persisted(command, "calibration_corrected_n5000_seed11.log")
assert BASIS_PATH.is_file(), f"Missing corrected basis: {BASIS_PATH}"
manifest = json.loads((CALIBRATION_ROOT / "run_manifest.json").read_text())
parity = json.loads((CALIBRATION_ROOT / "native_loss_gradient_parity.json").read_text())
assert manifest["state"] == "complete" and parity["status"] == "passed"
assert manifest["contract"] == "source_faithful_student_and_teacher_answer_colon_v2"
assert manifest["calibration_examples"] == 5000 and manifest["rank"] == 77
assert manifest["hidden_states"] == 13 and manifest["hidden_size"] == 768
print("Corrected basis SHA256:", manifest["basis_sha256"])

## 8. Run the all-state primary and layer-11 secondary screens

In [ ]:
ALL_STATES_ROOT = OUTPUT_ROOT / "utility_seed11" / "endpoint_all_states"
LAYER11_ROOT = OUTPUT_ROOT / "utility_seed11" / "endpoint_layer11"
if RUN_ALL_STATES_UTILITY:
    run_persisted(utility_command(ALL_STATES_ROOT, BASIS_PATH, "endpoint_all_states"), "utility_corrected_all_states_seed11.log")
if RUN_LAYER11_UTILITY:
    run_persisted(utility_command(LAYER11_ROOT, BASIS_PATH, "endpoint_layer11"), "utility_corrected_layer11_seed11.log")
for root in (ALL_STATES_ROOT, LAYER11_ROOT):
    utility_manifest = json.loads((root / "run_manifest.json").read_text())
    assert utility_manifest["state"] == "complete"
    assert len(utility_manifest["completed_batches"]) == 64
print("Both corrected scopes are complete")

## 9. Combine the preregistered decision

In [ ]:
from IPython.display import Markdown, display
REPORT_PATH = REPORT_ROOT / "official_codi_endpoint_tsvc_corrected_seed11.json"
run_persisted([sys.executable, "scripts/analyze_official_codi_endpoint_tsvc_corrected.py", "--all-states", str(ALL_STATES_ROOT), "--layer11", str(LAYER11_ROOT), "--output", str(REPORT_PATH)], "analyze_corrected_endpoint_tsvc_seed11.log")
report = json.loads(REPORT_PATH.read_text())
display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
print("FINAL STATUS:", report["status"])
print("TRAINING AUTHORIZED:", report["training_authorized"])

## 10. Build a checksummed corrected export

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_endpoint_tsvc_corrected_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(OUTPUT_ROOT, export_repo / "outputs" / "official_codi_endpoint_tsvc_corrected")
shutil.copytree(REPORT_ROOT, export_repo / "reports" / "official_codi_endpoint_tsvc_corrected")
shutil.copytree(LOG_ROOT, export_repo / "logs" / "official_codi_endpoint_tsvc_corrected")
validation_export = export_repo / "outputs" / "official_codi_gpt2_reproduction"
validation_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPRODUCTION_SUMMARY, validation_export / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text("Attach this corrected dataset and set RESUME_INPUT to its root. Historical endpoint TSV-C outputs are incompatible.\n")
files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
lines = [f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT_ROOT).as_posix()}" for path in files]
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n")
all_files = [path for path in EXPORT_ROOT.rglob("*") if path.is_file()]
print("Export root:", EXPORT_ROOT)
print("Files:", len(all_files), "Size MiB:", sum(path.stat().st_size for path in all_files) / 2**20)
print("Use Save Version with outputs enabled.")

## 11. Optional direct Kaggle Dataset upload

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, str(EXPORT_ROOT), version_notes=f"Corrected official CODI endpoint TSV-C gate at {commit}")
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct upload skipped; Save Version with outputs enabled is sufficient.")

## Interpretation

Only a passing `endpoint_all_states` gate authorizes a separately preregistered training study. A layer-11-only pass requires fresh confirmation. If both fail, close this corrected source-native linear rank-77 definition without tuning the observed contract.